# 02 Training curves

Loss, validation QWK and detection AUC per epoch for every finished run; the dashed line marks the selected epoch.

In [1]:
import json, os, sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd
from IPython.display import Image, display
pd.set_option("display.max_columns", 40, "display.width", 200, "display.precision", 4)
RESULTS, CHECKPOINTS = ROOT / "results", ROOT / "checkpoints"

def require(*paths, hint):
    missing = [str(p) for p in paths if not Path(p).exists()]
    if missing:
        print("MISSING:", *missing, sep="\n  ")
        print("To create:", hint)
    return not missing

def runs():
    return sorted(p for p in CHECKPOINTS.iterdir() if (p / "run.json").is_file()) if CHECKPOINTS.is_dir() else []


In [2]:
found = runs()
if not found:
    print("No finished run under checkpoints/. To create: python -m xaidr.cli train")
rows = []
for run in found:
    record = json.loads((run / "run.json").read_text())
    log = pd.DataFrame([json.loads(l) for l in (run / "metrics.jsonl").read_text().splitlines()])
    best = log.loc[log["epoch"] == record["best_epoch"]].iloc[0]
    rows.append({"run": run.name, "device": record["device"], "epochs_run": len(log),
                 "best_epoch": record["best_epoch"] + 1, "val_qwk": best["val_qwk"],
                 "val_det_auc": best["val_det_auc"], "selection_score": record["best_score"],
                 "threshold": record["threshold"], "hours": record["wall_clock_seconds"] / 3600})
display(pd.DataFrame(rows))

,run,device,epochs_run,best_epoch,val_qwk,val_det_auc,selection_score,threshold,hours
0,20261011-222646_convnext_tiny_oversample_seed42,cpu,20,18,0.9229,0.9981,0.9605,0.7896,5.6158
1,20261012-102317_convnext_tiny_none_seed42,cpu,5,5,0.9184,0.9992,0.9588,0.6567,1.0273
2,20261012-112520_convnext_tiny_oversample_seed42,cpu,5,5,0.9173,0.9983,0.9578,0.1104,1.7424


In [3]:
from xaidr.viz.figures import training_curves_fig
import matplotlib.pyplot as plt
for run in found:
    fig = training_curves_fig(run / "metrics.jsonl", run.name)
    display(fig); plt.close(fig)

<Figure size 1100x320 with 3 Axes>

<Figure size 1100x320 with 3 Axes>

<Figure size 1100x320 with 3 Axes>